In [215]:
import pymatgen as mg
import os
from natsort import natsorted
from pymatgen.io.vasp.inputs import Poscar
from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
import re
from itertools import groupby, permutations, product
from operator import itemgetter
import math

In [53]:
path = 'C:/Users/Adam/Desktop/voro-ml-si/datasets/icsd-all/'
targetFormula = 'AlTiNiN2O2'

composition = mg.Composition(targetFormula)
aFormula = composition.anonymized_formula
nElements = composition.to_data_dict['nelements']

In [54]:
fileList = natsorted(os.listdir(path))
total = fileList.__len__()


In [55]:
try:
    os.mkdir(path.split('/')[-2]+'-'+aFormula)
    print('Intermediate directory created.')
except FileExistsError:
    print('Intermediate directory already exist.')

Intermediate directory created.


In [56]:
allS=0
nEl=0
aF=0

for file in fileList:
    allS=allS+1
    if os.path.splitext(file)[1]=='':
        s = mg.Structure.from_file(path+file)
        if s.ntypesp==nElements:
            nEl=nEl+1
            if mg.Composition(s.formula).anonymized_formula==aFormula:
                formula = str(aF)+'-'+str.replace(s.formula,' ','')
                s = Poscar(s)
                s.write_file(path.split('/')[-2]+'-'+aFormula+'/'+formula+'.POSCAR')
                aF=aF+1
                if aF%100==0:
                    print(str(aF)+' files identified and converted.')
    if allS%5000==0:
        print(str(round(allS/total*100,1))+'% database scan completed.')

print('\nTotal scanned: '+str(allS))
print('\n'+str(nElements)+'-element structures: '+str(nEl))
print(str(nElements)+'-element structures fraction: '+str(round(nEl/total*100))+'%')
print('\n'+aFormula+': '+str(aF))
print(aFormula+' Fraction from '+str(nElements)+'-element: '+str(round(aF/nEl*100))+'%')


15.5% database scan completed.
31.0% database scan completed.
46.5% database scan completed.
62.0% database scan completed.
77.5% database scan completed.
93.0% database scan completed.

Total scanned: 32258

5-element structures: 941
5-element structures fraction: 3%

ABCD2E2: 11
ABCD2E2 Fraction from 5-element: 1%


In [57]:
candidateList = natsorted(os.listdir(path.split('/')[-2]+'-'+aFormula))

In [58]:
candidateList[0]

'0-Ag4H4I2O2F2.POSCAR'

In [59]:
s = mg.Structure.from_file(path.split('/')[-2]+'-'+aFormula+'/'+candidateList[0])

In [71]:
fractions = re.sub(r'[A-z]+', '', s.formula).split()
elements = re.sub(r'[0-9]+', '', s.formula).split()
print(fractions)
print(elements)

['4', '4', '2', '2', '2']
['Ag', 'H', 'I', 'O', 'F']


In [68]:
fractions = re.sub(r'[A-z]+', '', composition.formula).split()
elements = re.sub(r'[0-9]+', '', composition.formula).split()
print(fractions)
print(elements)

['1', '1', '1', '2', '2']
['Ti', 'Al', 'Ni', 'N', 'O']


In [72]:
compl = []
for i in range(elements.__len__()):
    compl.append([elements[i],int(fractions[i])])
print(compl)

[{'element': 'Ag', 'fraction': '4'}, {'element': 'H', 'fraction': '4'}, {'element': 'I', 'fraction': '2'}, {'element': 'O', 'fraction': '2'}, {'element': 'F', 'fraction': '2'}]
[['Ag', 4], ['H', 4], ['I', 2], ['O', 2], ['F', 2]]


In [203]:
lki = compl
lki.sort(key=itemgetter(1))

glo = [[x for x,y in g]
       for k,g in  groupby(lki,key=itemgetter(1))]

print(glo)

[['I', 'O', 'F'], ['Ag', 'H']]


In [172]:
somelist=[]
for lev in glo:
    tList = []
    ptL = list(permutations(lev))
    for i in ptL:
        tList.append(list(i))
    somelist.append(tList)

In [173]:
for element in product(*somelist):
    print(list(element))
print()

[['I', 'O', 'F'], ['Ag', 'H']]
[['I', 'O', 'F'], ['H', 'Ag']]
[['I', 'F', 'O'], ['Ag', 'H']]
[['I', 'F', 'O'], ['H', 'Ag']]
[['O', 'I', 'F'], ['Ag', 'H']]
[['O', 'I', 'F'], ['H', 'Ag']]
[['O', 'F', 'I'], ['Ag', 'H']]
[['O', 'F', 'I'], ['H', 'Ag']]
[['F', 'I', 'O'], ['Ag', 'H']]
[['F', 'I', 'O'], ['H', 'Ag']]
[['F', 'O', 'I'], ['Ag', 'H']]
[['F', 'O', 'I'], ['H', 'Ag']]



In [174]:
def sortedElementsList(mgComp):
    fractions = re.sub(r'[A-z]+', '', mgComp.formula).split()
    elements = re.sub(r'[0-9]+', '', mgComp.formula).split()
    compl = []
    for i in range(elements.__len__()):
        compl.append([elements[i],int(fractions[i])])
    compl.sort(key=itemgetter(1))
    sortedEL = [[x for x,y in g]
           for k,g in  groupby(compl,key=itemgetter(1))]
    return sortedEL

In [194]:
sortedElementsList(mg.Composition('Al2Ni3O2U4As3I4Bi2'))

[['Al', 'Bi', 'O'], ['Ni', 'As'], ['U', 'I']]

In [224]:
sortedElementsList(mg.Composition('K2Hg2C4I2N4'))

[['K', 'Hg', 'I'], ['C', 'N']]

In [212]:
def elementPermutations(sEL):
    generatingList=[]
    outList = []
    for lev in sEL:
        tList = []
        ptL = list(permutations(lev))
        for i in ptL:
            tList.append(list(i))
        generatingList.append(tList)
    for permutation in product(*generatingList):
        outList.append([item for sublist in list(permutation) for item in sublist])
    return outList

In [223]:
elementPermutations(sortedElementsList(mg.Composition('K2Hg2C4I2N4')))


[['K', 'Hg', 'I', 'C', 'N'],
 ['K', 'Hg', 'I', 'N', 'C'],
 ['K', 'I', 'Hg', 'C', 'N'],
 ['K', 'I', 'Hg', 'N', 'C'],
 ['Hg', 'K', 'I', 'C', 'N'],
 ['Hg', 'K', 'I', 'N', 'C'],
 ['Hg', 'I', 'K', 'C', 'N'],
 ['Hg', 'I', 'K', 'N', 'C'],
 ['I', 'K', 'Hg', 'C', 'N'],
 ['I', 'K', 'Hg', 'N', 'C'],
 ['I', 'Hg', 'K', 'C', 'N'],
 ['I', 'Hg', 'K', 'N', 'C']]

In [220]:
candidateList = natsorted(os.listdir(path.split('/')[-2]+'-'+aFormula))

sEL = sortedElementsList(mg.Composition(targetFormula))
sELtarget = [item for sublist in sEL for item in sublist]
print('Target Element Combination: '+ str(sELtarget))

permN = 1
for multiples in sEL:
    permN = permN*math.factorial(multiples.__len__())

if permN>1:
    print('\nTarget composition has degenerate anonymous formula thus requires permutations.')
    print('Up to '+str(permN)+' per structure.')
    print('Total number of candidates increased up to: '+str(aF*permN)+'\n')

intPath = path.split('/')[-2]+'-'+aFormula
print('Intermediate Directory: '+intPath)
finPath = path.split('/')[-2]+'-'+targetFormula
print('Final Directory: '+finPath)
try:
    os.mkdir(finPath)
    print('Final directory created.')
except FileExistsError:
    print('Final directory already exist.')

allS=0
for candidate in candidateList:
    s = mg.Structure.from_file(intPath+'/'+candidate)

    permuts = elementPermutations(sortedElementsList(mg.Composition(s.formula)))
    spGroupA = SpacegroupAnalyzer(s)
    fileSplit = candidate.split('.')
    newNameRoot = fileSplit[0]+'-'+spGroupA.get_crystal_system()+'-'+str(spGroupA.get_space_group_number())+'-'

    p=0
    for permut in permuts:
        p=p+1
        st = s.copy()
        for e in range(permut.__len__()):
            st.replace_species({mg.Element(permut[e]): mg.Element(sELtarget[e])})
        if permuts.__len__()>1:
            newName = newNameRoot+str.replace(st.formula,' ','')+'-permutation'+str(p)+'.POSCAR'
        else:
            newName = newNameRoot+str.replace(st.formula,' ','')+'.POSCAR'
        sP = Poscar(st)
        sP.write_file(finPath+'/'+newName)
    allS = allS+p

print('\nGenerated '+str(allS)+' '+targetFormula+' candidates.')


Target Element Combination: ['Ti', 'Al', 'Ni', 'N', 'O']

Target composition has degenerate anonymous formula thus requires permutations.
Up to 12 per structure.
Total number of candidates increased up to: 132

Intermediate Directory: icsd-all-ABCD2E2
Final Directory: icsd-all-AlTiNiN2O2
Final directory already exist.

Generated 132 AlTiNiN2O2 candidates.


In [221]:
targetFormula = 'AlTiNiN2O2'

composition = mg.Composition(targetFormula)
fractions = re.sub(r'[A-z]+', '', composition.formula).split()
elements = re.sub(r'[0-9]+', '', composition.formula).split()
print(fractions)
print(elements)


['1', '1', '1', '2', '2']
['Ti', 'Al', 'Ni', 'N', 'O']


In [236]:
candidate = '1-K2Hg2C4I2N4.POSCAR'
s = mg.Structure.from_file(intPath+'/'+candidate)

permuts = elementPermutations(sortedElementsList(mg.Composition(s.formula)))
spGroupA = SpacegroupAnalyzer(s)
fileSplit = candidate.split('.')
newNameRoot = fileSplit[0]+'-'+spGroupA.get_crystal_system()+'-'+str(spGroupA.get_space_group_number())+'-'
print(newNameRoot)

for permut in permuts:
    d={}
    p=p+1
    st = s.copy()
    print(st)
    for e in range(permut.__len__()):
        d.update({mg.Element(permut[e]): mg.Element(sELtarget[e])})
        print(d)
    st.replace_species(d)
    print(st)

    if permuts.__len__()>1:
        newName = newNameRoot+str.replace(st.formula,' ','')+'-permutation'+str(p)+'.POSCAR'
    else:
        newName = newNameRoot+str.replace(st.formula,' ','')+'.POSCAR'
    sP = Poscar(st)
    sP.write_file(finPath+'/'+newName)

1-K2Hg2C4I2N4-orthorhombic-63-
Full Formula (K2 Hg2 C4 I2 N4)
Reduced Formula: KHgC2IN2
abc   :   4.414778   8.432376   9.691811
angles:  90.000000  90.000000  74.824563
Sites (14)
  #  SP           a         b         c
---  ----  --------  --------  --------
  0  C     0.877525  0.244951  0.050688
  1  C     0.877525  0.244951  0.449311
  2  C     0.122476  0.755049  0.550689
  3  C     0.122476  0.755049  0.949312
  4  Hg    0         0         0
  5  Hg    0         0         0.5
  6  I     0.552016  0.895969  0.25
  7  I     0.447985  0.104031  0.75
  8  K     0.236026  0.527949  0.25
  9  K     0.763975  0.472051  0.75
 10  N     0.808453  0.383096  0.084484
 11  N     0.808453  0.383096  0.415516
 12  N     0.191548  0.616904  0.584484
 13  N     0.191548  0.616904  0.915516
{Element K: Element Ti}
{Element K: Element Ti, Element Hg: Element Al}
{Element K: Element Ti, Element Hg: Element Al, Element I: Element Ni}
{Element K: Element Ti, Element Hg: Element Al, Element I: Eleme

In [235]:
d = {'a':'b'}
d2 = d.update({'c':'d'})
print(d)

{'a': 'b', 'c': 'd'}
